# 02 · Turbulence Audit
Manual tau replication, chi2(N) calibration, decomposition, vol-standardization.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from scipy import stats as sp_stats

from data.synthetic import generate_em_universe
from modules.turbulence import compute_turbulence_index, crisis_episodes
from core.covariance import SlowCovariance, VolStandardizer

uni = generate_em_universe(seed=42)
panel  = uni.panel
fx_ret = uni.fx
eq_ret = uni.equity

WINDOW = 252; SLOW_W = 504

turb_vs  = compute_turbulence_index(
    panel, window=WINDOW, min_periods=60, vol_standardize=True,  slow_window=SLOW_W)
turb_raw = compute_turbulence_index(
    panel, window=WINDOW, min_periods=60, vol_standardize=False, slow_window=SLOW_W)

print(f'Panel shape  : {panel.shape}')
print(f'Current regime (vol-std): {turb_vs.current_regime()}')
print(f'Current score (vol-std) : {turb_vs.current_score():.2f}')


## 2 · Manual tau Replication for 2020-03-20

tau_t = (r_t - mu)' Sigma^{-1} (r_t - mu),  Sigma = SLOW-track covariance.


In [ ]:
TARGET = '2020-03-20'
loc = panel.index.get_indexer([TARGET], method='nearest')[0]
audit_idx = panel.index[loc]
print(f'Audit date: {audit_idx.date()}')

# estimation window
window_data = panel.iloc[max(0, loc - SLOW_W):loc].dropna(how='any')
print(f'Window: {len(window_data)} obs  ({window_data.index[0].date()} to {window_data.index[-1].date()})')

cov_result = SlowCovariance().fit(window_data, window=SLOW_W)
mu        = cov_result.mu
Sigma_inv = cov_result.sigma_inv

# vol-standardized returns
vol_std_panel = VolStandardizer().fit_transform(panel).fillna(0.0)
r_t = vol_std_panel.loc[audit_idx].values

diff       = r_t - mu
tau_manual = float(diff @ Sigma_inv @ diff)
tau_module = float(turb_vs.turbulence.loc[audit_idx])

print(f'tau (manual) : {tau_manual:.4f}')
print(f'tau (module) : {tau_module:.4f}')
print(f'abs diff     : {abs(tau_manual - tau_module):.6f}')
print(f'Regime       : {turb_vs.regime.loc[audit_idx]}')


## 3 · chi2(N=14) Calibration Plot

Under H0 (multivariate normality) tau ~ chi2(N).
Empirical CDF vs theoretical — sustained deviation flags fat tails.


In [ ]:
from statsmodels.distributions.empirical_distribution import ECDF

N = panel.shape[1]

CALM_MASK = (
    ((panel.index >= '2015-01-01') & (panel.index < '2018-06-01')) |
    ((panel.index >= '2021-01-01') & (panel.index < '2022-01-01'))
)
tau_calm = turb_vs.turbulence.loc[CALM_MASK].dropna()

fig, ax = plt.subplots(figsize=(8, 5))
x = np.linspace(0, float(tau_calm.quantile(0.999)), 300)
ax.plot(x, sp_stats.chi2.cdf(x, df=N), 'r--', lw=1.5, label=f'Theoretical chi2({N})')
ecdf = ECDF(tau_calm.values)
ax.plot(x, ecdf(x), 'b-', lw=1.5, label='Empirical (calm periods)')
ax.axvline(sp_stats.chi2.ppf(0.95, df=N), color='orange', ls=':', alpha=0.7, label='chi2(0.95)')
ax.axvline(sp_stats.chi2.ppf(0.99, df=N), color='red',    ls=':', alpha=0.7, label='chi2(0.99)')
ax.set_xlabel('tau'); ax.set_ylabel('CDF')
ax.set_title(f'Empirical vs chi2({N}) CDF — calm-period turbulence')
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()
print(f'Calm tau: mean={tau_calm.mean():.2f}  median={tau_calm.median():.2f}')
print(f'chi2({N}) theoretical mean: {N}')


## 4 · Turbulence Time Series with Regime Shading


In [ ]:
REGIME_HEX = {'Calm':'#2ecc71','Elevated':'#f39c12','Turbulent':'#e67e22','Crisis':'#e74c3c'}

fig, ax = plt.subplots(figsize=(14, 4))
ts  = turb_vs.turbulence.dropna()
reg = turb_vs.regime.reindex(ts.index)

for label, col in REGIME_HEX.items():
    mask = reg == label
    if not mask.any(): continue
    in_block, x0 = False, None
    for dt, val in mask.items():
        if val and not in_block:  x0, in_block = dt, True
        elif not val and in_block:
            ax.axvspan(x0, dt, alpha=0.25, color=col, lw=0)
            in_block = False
    if in_block: ax.axvspan(x0, mask.index[-1], alpha=0.25, color=col, lw=0)

ax.plot(ts.index, ts.values, color='#00d4aa', lw=0.9)
for thr, label in [('elevated','Elevated'),('turbulent','Turbulent'),('crisis','Crisis')]:
    ax.axhline(turb_vs.thresholds[thr], color=REGIME_HEX[label], ls='--', lw=0.8, alpha=0.7)
patches = [mpatches.Patch(color=REGIME_HEX[l], alpha=0.5, label=l) for l in REGIME_HEX]
ax.legend(handles=patches, loc='upper left', fontsize=8)
ax.set_ylabel('tau (Mahalanobis2, vol-std)'); ax.grid(alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title('Turbulence Index — Full Panel (vol-standardized)')
plt.tight_layout(); plt.show()


## 5 · Magnitude vs Correlation Decomposition


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

mag  = turb_vs.magnitude_component.reindex(ts.index)
corr = turb_vs.correlation_component.reindex(ts.index)

axes[0].fill_between(ts.index, mag,  alpha=0.7, color='#3498db', label='Magnitude')
axes[0].fill_between(ts.index, corr, alpha=0.7, color='#9b59b6', label='Correlation')
axes[0].set_ylabel('Component tau'); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.15)
axes[0].set_title('Turbulence Decomposition — Magnitude vs Correlation component')

total    = (mag + corr).clip(lower=1e-8)
frac_cor = corr.clip(lower=0) / total
axes[1].plot(frac_cor.index, frac_cor.rolling(21).mean(), color='#9b59b6', lw=1.1)
axes[1].axhline(0.5, color='white', ls='--', lw=0.8, alpha=0.4)
axes[1].set_ylabel('Fraction from correlation (21d MA)'); axes[1].grid(alpha=0.15)
axes[1].set_ylim(0, 1)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()


## 6 · Crisis Episode Table


In [ ]:
for regime_label, min_dur in [('Crisis', 3), ('Turbulent', 5)]:
    eps = crisis_episodes(turb_vs, regime=regime_label, min_duration_days=min_dur)
    print(f'{regime_label} episodes (>={min_dur}d): {len(eps)}')
    if not eps.empty:
        eps['start'] = pd.to_datetime(eps['start']).dt.strftime('%Y-%m-%d')
        eps['end']   = pd.to_datetime(eps['end']).dt.strftime('%Y-%m-%d')
        eps.columns  = ['Start','End','Duration (d)','Peak tau','Mean tau']
        print(eps.round(2).to_string(index=False))
    print()


## 7 · Vol-Standardization Effect

A pure vol spike (no correlation change) should NOT register as Crisis
under vol-standardization.  EWMA vol adapts within ~30 days.


In [ ]:
loc_spike = panel.index.get_indexer(['2020-03-20'], method='nearest')[0]
spike_date = panel.index[loc_spike]
panel_spiked = panel.copy()
panel_spiked.loc[spike_date] *= 5.0

t_vs  = compute_turbulence_index(panel_spiked, window=WINDOW, min_periods=60,
                                  vol_standardize=True,  slow_window=SLOW_W)
t_raw = compute_turbulence_index(panel_spiked, window=WINDOW, min_periods=60,
                                  vol_standardize=False, slow_window=SLOW_W)

print(f'Spike date : {spike_date.date()}')
print(f'  vol-std  tau={t_vs.turbulence.loc[spike_date]:.1f}  regime={t_vs.regime.loc[spike_date]}')
print(f'  raw      tau={t_raw.turbulence.loc[spike_date]:.1f}  regime={t_raw.regime.loc[spike_date]}')
if t_vs.regime.loc[spike_date] != 'Crisis':
    print('OK: vol-standardization suppressed the pure vol spike')
else:
    print('NOTE: spike still Crisis — EWMA adaptation speed may need tuning')
